# 🚗 Detecció de Matrícules — Harris + Shi-Tomasi + Density Map

```
Imatge RGB
    ▼
[1] Preprocessament: gris → Gaussià → CLAHE
    ▼
[2] Harris Corners  ──┐
                      ├─► fusió + NMS local
[3] Shi-Tomasi      ──┘
    ▼
[4] Mapa de densitat (Gaussian kernel sobre punts corners)
    ▼
[5] Binarització adaptativa (Otsu sobre el mapa de densitat)
    ▼
[6] Morfologia: closing horitzontal + opening
    ▼
[7] findContours → bounding box per contorn
    ▼
[8] Filtre per forma (AR, àrea, mida mínima)
    ▼
Sortida: N bounding boxes candidates
```

**Idea clau**: en lloc de passar els corners directament a un clusterer, construïm un **mapa de calor** on cada píxel codifica la densitat local de corners. Les regions amb text (matrícules) generen pics de densitat locals i compactes. Després binaritzem amb Otsu (adaptatiu per imatge) i usem `findContours` per extreure regions.

## 1. Imports

In [ ]:
import random
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(42); np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['image.cmap'] = 'gray'

## 2. Configuració

In [ ]:
RAW_DIR    = Path('data/raw')
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
N_SAMPLES  = 108

# --- Harris params ---
HARRIS_K        = 0.04
HARRIS_KSIZE    = 3
HARRIS_THRESH   = 0.001   # lower = more corners detected

# --- Shi-Tomasi params ---
ST_MAX_CORNERS  = 600
ST_QUALITY      = 0.005
ST_MIN_DIST     = 3

# --- NMS on corner points ---
NMS_RADIUS      = 5       # suppress corners within this radius (px)

# --- Density map ---
DENSITY_SIGMA   = 6       # gaussian kernel sigma — smaller = more local peaks
                          # increase if plate corners are sparse

# --- Morphology on binary density map ---
CLOSE_W, CLOSE_H = 15, 3  # horizontal closing to merge character blobs

# --- Shape filter ---
AREA_RATIO_MIN  = 0.0005
AREA_RATIO_MAX  = 0.12
ASPECT_MIN      = 1.5
ASPECT_MAX      = 9.0
MIN_WIDTH       = 20
MIN_HEIGHT      = 6
MIN_CONTOUR_AREA = 60

## 3. Preprocessament

In [ ]:
def preprocess(img_bgr):
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.bilateralFilter(gray, 9, 75, 75)
    #blurred = cv2.GaussianBlur(gray, (5, 5), sigmaX=1.0)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(blurred)

## 4. Detecció de corners (Harris + Shi-Tomasi)

**Harris**: $R = \det(M) - k \cdot \text{tr}(M)^2$ on $M = \sum_{W} w \begin{bmatrix} I_x^2 & I_xI_y \\ I_xI_y & I_y^2 \end{bmatrix}$. Bona detecció de cantonades netes.

**Shi-Tomasi**: $R = \min(\lambda_1, \lambda_2)$. Més estable en regions de textura. Complementari a Harris — junts cobreixen millor la matrícula.

In [ ]:
def detect_harris(gray):
    r      = cv2.cornerHarris(np.float32(gray), blockSize=2,
                              ksize=HARRIS_KSIZE, k=HARRIS_K)
    ys, xs = np.where(r > HARRIS_THRESH * r.max())
    pts    = np.column_stack([xs, ys]).astype(np.float32)
    return pts, r[ys, xs]

def detect_shi_tomasi(gray):
    p = cv2.goodFeaturesToTrack(gray, ST_MAX_CORNERS, ST_QUALITY, ST_MIN_DIST)
    if p is None:
        return np.empty((0, 2), np.float32), np.empty(0)
    pts = p.reshape(-1, 2)
    return pts, np.ones(len(pts))

## 5. NMS local sobre corners

Suprimim corners redundants dins un radi `NMS_RADIUS`. Quedem-nos amb el de major resposta. Redueix soroll i accelera el pas de densitat.

In [ ]:
def nms_corners(pts, scores, radius=NMS_RADIUS):
    if len(pts) == 0:
        return pts, scores
    order          = np.argsort(scores)[::-1]
    pts, scores    = pts[order], scores[order]
    keep           = np.ones(len(pts), dtype=bool)
    for i in range(len(pts)):
        if not keep[i]: continue
        dists = np.linalg.norm(pts[i+1:] - pts[i], axis=1)
        keep[i+1:][dists < radius] = False
    return pts[keep], scores[keep]

def merge_corners(gray):
    # Detect with both methods, NMS each, then merge.
    ph, sh = detect_harris(gray);     ph, sh = nms_corners(ph, sh)
    ps, ss = detect_shi_tomasi(gray); ps, ss = nms_corners(ps, ss)
    if len(ph) > 0 and len(ps) > 0:
        return np.vstack([ph, ps])
    return ph if len(ph) > 0 else ps

def merge_corners2(gray):
    pts, scores = detect_harris(gray)
    pts, scores = nms_corners(pts, scores)
    return pts

## 6. Mapa de densitat de corners

Creem una imatge buida i posem un '1' a cada posició de corner. Després apliquem un **blur Gaussià** amb $\sigma$ = `DENSITY_SIGMA`. El resultat és un mapa de calor on les regions amb molts corners propers brillen més — la matrícula és exactament una d'aquestes regions.

No normalitzem a [0,1] sinó que conservem valors absoluts perquè Otsu pugui distingir pics de fons correctament.

In [ ]:
def build_density_map(pts, img_shape, sigma=DENSITY_SIGMA):
    H, W  = img_shape[:2]
    d     = np.zeros((H, W), dtype=np.float32)
    for x, y in pts:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= xi < W and 0 <= yi < H:
            d[yi, xi] += 1.0
    ksize = int(6 * sigma + 1) | 1   # odd kernel size
    return cv2.GaussianBlur(d, (ksize, ksize), sigmaX=sigma)

## 7. Binarització + morfologia + findContours

**Otsu** sobre el mapa de densitat tria automàticament el llindar que millor separa regions d'alta densitat (corners) de les de baixa densitat (fons).

Apliquem un **closing horitzontal** perquè els caràcters individuals de la matrícula generen pics separats que el closing fusiona en un sol blob.

In [ ]:
def density_to_boxes(density, img_shape):
    H, W     = img_shape[:2]
    img_area = H * W

    # Normalise to uint8 for Otsu
    d8 = cv2.normalize(density, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, binary = cv2.threshold(d8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Elimina estructures verticals fines (logos, antenes, etc.)
    #k_open_vert = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 25))
    #binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k_open_vert)

    # Horizontal closing: merge per-character blobs into a single plate blob
    k_close = cv2.getStructuringElement(cv2.MORPH_RECT, (CLOSE_W, CLOSE_H))
    binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close)

    # Small opening: remove isolated noise blobs
    k_open = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k_open)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) < MIN_CONTOUR_AREA: continue
        x, y, w, h = cv2.boundingRect(cnt)
        if w < MIN_WIDTH or h < MIN_HEIGHT:                          continue
        if not (AREA_RATIO_MIN <= w*h / img_area <= AREA_RATIO_MAX): continue
        if not (ASPECT_MIN <= w / float(h) <= ASPECT_MAX):           continue
        boxes.append((x, y, w, h))

    return binary, boxes

## 8. Pipeline completa

In [ ]:
def detect_plates_corners(img_bgr):
    enhanced = preprocess(img_bgr)
    pts      = merge_corners(enhanced)
    density  = build_density_map(pts, img_bgr.shape)
    binary, boxes = density_to_boxes(density, img_bgr.shape)
    return {
        'enhanced': enhanced,
        'pts':      pts,
        'density':  density,
        'binary':   binary,
        'boxes':    boxes,
    }

## 9. Visualització (totes les fases)

In [ ]:
def show_stages(img_bgr, result, title=''):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    axes[0,0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0,0].set_title('Original'); axes[0,0].axis('off')

    axes[0,1].imshow(result['enhanced'])
    axes[0,1].set_title('1. Preprocessat'); axes[0,1].axis('off')

    # Corners overlaid on enhanced
    vis = cv2.cvtColor(result['enhanced'], cv2.COLOR_GRAY2RGB)
    if len(result['pts']) > 0:
        for x, y in result['pts'].astype(int):
            if 0 <= x < vis.shape[1] and 0 <= y < vis.shape[0]:
                vis[y, x] = [255, 255, 0]
    axes[0,2].imshow(vis)
    axes[0,2].set_title(f"2+3. Corners Harris+ST NMS ({len(result['pts'])})"); axes[0,2].axis('off')

    d_disp = cv2.normalize(result['density'], None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    axes[1,0].imshow(d_disp, cmap='hot')
    axes[1,0].set_title(f'4. Density map (σ={DENSITY_SIGMA})'); axes[1,0].axis('off')

    axes[1,1].imshow(result['binary'], cmap='gray')
    axes[1,1].set_title('5+6. Otsu + closing + findContours'); axes[1,1].axis('off')

    axes[1,2].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    for (x, y, w, h) in result['boxes']:
        axes[1,2].add_patch(patches.Rectangle((x,y), w, h, linewidth=2,
                            edgecolor='lime', facecolor='none'))
    axes[1,2].set_title(f"7. Final boxes ({len(result['boxes'])})"); axes[1,2].axis('off')

    plt.tight_layout(); plt.show()

## 10. Execució sobre 10 imatges aleatòries

In [ ]:
all_images     = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in VALID_EXTS])
print(f'Found {len(all_images)} images')
selected_paths = random.sample(all_images, min(N_SAMPLES, len(all_images)))

results = []
for path in selected_paths:
    img = cv2.imread(str(path))
    if img is None: continue
    res = detect_plates_corners(img)
    results.append((path, img, res))
    print(f"{path.name:30s}  corners={len(res['pts']):4d}  final_boxes={len(res['boxes']):2d}")

In [ ]:
def debug_morphology(img_bgr):
    enhanced = preprocess(img_bgr)
    pts      = merge_corners(enhanced)
    density  = build_density_map(pts, img_bgr.shape)

    # --- Otsu ---
    d8 = cv2.normalize(density, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, after_otsu = cv2.threshold(d8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # --- Opening vertical ---
    k_open_vert  = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15))
    after_open   = cv2.morphologyEx(after_otsu, cv2.MORPH_OPEN, k_open_vert)

    # --- Closing horitzontal ---
    k_close_horiz = cv2.getStructuringElement(cv2.MORPH_RECT, (CLOSE_W, CLOSE_H))
    after_close   = cv2.morphologyEx(after_open, cv2.MORPH_CLOSE, k_close_horiz)

    # --- Plot ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(img_path.name, fontsize=13, fontweight='bold')

    axes[0].imshow(after_otsu,  cmap='gray'); axes[0].set_title('Otsu');              axes[0].axis('off')
    axes[1].imshow(after_open,  cmap='gray'); axes[1].set_title('+ Open vertical');   axes[1].axis('off')
    axes[2].imshow(after_close, cmap='gray'); axes[2].set_title('+ Close horitzontal'); axes[2].axis('off')

    plt.tight_layout(); plt.show()


# Executa sobre les imatges seleccionades
#for img_path in selected_paths:
    #img = cv2.imread(str(img_path))
    #if img is not None:
    #    debug_morphology(img)

In [ ]:
for path, img, res in results:
    show_stages(img, res, title=path.name)

## 11. Vista resum

In [ ]:
n = len(results); cols = 2; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, 5*rows))
axes = np.atleast_2d(axes).flatten()
for ax, (path, img, res) in zip(axes, results):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for (x, y, w, h) in res['boxes']:
        ax.add_patch(patches.Rectangle((x,y), w, h, linewidth=2,
                     edgecolor='lime', facecolor='none'))
    ax.set_title(f"{path.name} -- {len(res['boxes'])} boxes")
    ax.axis('off')
for ax in axes[len(results):]: ax.axis('off')
plt.tight_layout(); plt.show()